# Preparação, anonimização e curadoria dos dados

Este caderno organiza exemplos institucionais sintéticos para ajuste fino supervisionado. A preparação separa conteúdo, instrução e resposta esperada, remove identificadores diretos e preserva a origem de cada exemplo. Nenhum dado real é utilizado.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data/raw/internal_examples.jsonl"
records = [json.loads(line) for line in RAW.read_text(encoding="utf-8").splitlines() if line]
print(f"Exemplos brutos: {len(records)}")
print("Categorias:", sorted({item["category"] for item in records}))

## Inspeção inicial

A inspeção procura campos ausentes, duplicidades, respostas muito curtas e distribuição desequilibrada. O exemplo exibido é sintético e contém identificadores deliberados para testar a anonimização.

In [ ]:
example = next(item for item in records if item["id"] == "SEC-001")
example

## Anonimização

Expressões regulares removem CPF, e-mail, telefone, números de prontuário e nomes ligados a rótulos. O identificador `PAC-0000` é sintético e necessário para a ligação controlada com a base estruturada.

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from clinical_assistant.anonymization import anonymize_text

result = anonymize_text(example["input"])
print(result.text)
print(result.redactions)

## Formatação e divisão

Cada registro é convertido para um formato instrucional causal. A separação reserva uma observação de cada categoria para teste, evitando que a avaliação omita um tipo de tarefa.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts/prepare_data.py")], cwd=ROOT, check=True)
report = json.loads((ROOT / "data/processed/curation_report.json").read_text(encoding="utf-8"))
report

In [ ]:
train = [json.loads(line) for line in (ROOT / "data/processed/train.jsonl").read_text(encoding="utf-8").splitlines()]
test = [json.loads(line) for line in (ROOT / "data/processed/test.jsonl").read_text(encoding="utf-8").splitlines()]
print("Treino:", len(train), "| Teste:", len(test))
print(train[0]["text"])

## Resultado da curadoria

O relatório registra tamanho, categorias, fontes e resultado da verificação de identificadores. Como o corpus é pequeno, sua função é demonstrar o processo e adaptar comportamento; não sustenta generalização clínica.